# 02 · Robustness: does the protection survive WhatsApp / Instagram?

**Goal (Review-2):** Review-1 showed PGD breaks StarGAN on the exact protected pixels.
But a real photo is **saved, compressed and resized** before a deepfaker gets it.
This notebook answers two questions:

| Part | Question | Roadmap task |
|---|---|---|
| A | How much of the Review-1 (plain PGD) protection survives JPEG / resize / blur / WhatsApp-like sharing? | 2.1 |
| B | Does **EoT-PGD** (optimising *through* a differentiable JPEG) keep the protection alive? | 2.2, 2.3 |
| C | Side-by-side table + figures for Review-2 | 2.5 |
| D | **Real test:** send the photos through actual WhatsApp / Instagram and measure again | 2.7 |

How a shared photo is scored: we compare the fake made from the **protected** photo after
sharing against the fake made from the **unprotected** photo shared the same way. So
compression's own small effect never counts as "protection".

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Needs the
`KAGGLE_API_TOKEN` secret, same as notebook 01.
**Runtime:** Part A ≈ 8 min, Part B ≈ 40 min for 20 faces × 2 ε on a T4.

## 1 · Setup (same as notebook 01)

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
!git clone -q https://github.com/Liya-17/immunize.git
%cd immunize
!git clone -q https://github.com/yunjey/stargan.git stargan_official
!pip install -q kagglehub scikit-image
!python tests/test_diff_jpeg.py      # checks our differentiable JPEG against the real one

In [ ]:
import os, kagglehub
import pandas as pd
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
CELEBA = kagglehub.dataset_download("jessicali9530/celeba-dataset")
IMG_DIR = f"{CELEBA}/img_align_celeba/img_align_celeba"
parts = pd.read_csv(f"{CELEBA}/list_eval_partition.csv")
attrs = pd.read_csv(f"{CELEBA}/list_attr_celeba.csv").set_index("image_id")

N_IMAGES = 20                                   # same 20 test faces as Review-1
test_ids = parts[parts["partition"] == 2]["image_id"].head(N_IMAGES).tolist()
SELECTED = ["Black_Hair", "Blond_Hair", "Brown_Hair", "Male", "Young"]
labels = ((attrs.loc[test_ids, SELECTED].values + 1) // 2).astype("float32")
print(len(test_ids), "test faces")

In [ ]:
!mkdir -p weights
!wget -q -O weights/stargan256.zip "https://www.dropbox.com/s/zdq6roqf63m0v5f/celeba-256x256-5attrs.zip?dl=1"
!unzip -q -o weights/stargan256.zip -d weights/stargan256

import glob, sys
import torch.nn as nn
from PIL import Image
from torchvision import transforms as T
sys.path.insert(0, "stargan_official")
from model import Generator

device = "cuda" if torch.cuda.is_available() else "cpu"
G = Generator(conv_dim=64, c_dim=5, repeat_num=6).to(device)
ckpt = sorted(glob.glob("weights/stargan256/**/*-G.ckpt", recursive=True))
G.load_state_dict(torch.load(ckpt[-1], map_location=device)); G.eval()
for p in G.parameters():
    p.requires_grad_(False)

EDIT_NAMES = ["Black hair", "Blond hair", "Brown hair", "Flip gender", "Flip age"]

def target_labels(c_org):
    targets = []
    for i in range(5):
        c = c_org.clone()
        if i < 3:
            c[:3] = 0; c[i] = 1
        else:
            c[i] = 1 - c[i]
        targets.append(c)
    return torch.stack(targets)

class StarGANEdits(nn.Module):
    def __init__(self, G, c_targets):
        super().__init__(); self.G, self.c = G, c_targets
    def forward(self, x):
        x = x.expand(len(self.c), -1, -1, -1) * 2 - 1
        return (self.G(x, self.c) + 1) / 2

preprocess = T.Compose([T.CenterCrop(178), T.Resize(256), T.ToTensor()])
def load_face(image_id):
    return preprocess(Image.open(f"{IMG_DIR}/{image_id}").convert("RGB")).unsqueeze(0).to(device)

def face_and_model(i):
    return load_face(test_ids[i]), StarGANEdits(G, target_labels(torch.tensor(labels[i])).to(device))
print("StarGAN loaded:", ckpt[-1])

## 2 · The sharing transforms we test

All are the **real** operations (PIL's libjpeg encoder, antialiased resize), defined in
`eval/robustness.py`. "WhatsApp-like" = downscale 75 % + JPEG q=70 (4:2:0);
"Instagram-like" = resample 90 % + JPEG q=80; "Forwarded twice" = WhatsApp-like applied twice.
Part D replaces these approximations with the real apps.

In [ ]:
import time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

from attack.pgd import pgd_disrupt
from attack.eot import eot_pgd_disrupt
from eval.metrics import psnr, ssim
from eval.robustness import TRANSFORMS, survival_rows, quantize_8bit, MSE_SUCCESS

os.makedirs("results", exist_ok=True); os.makedirs("docs/figures", exist_ok=True)
EPS_LIST = (4/255, 8/255)
COLS = [(f"{round(e*255)}/255", m) for e in EPS_LIST for m in ("PGD", "EoT-PGD")]   # table column order
PGD_STEPS = 40                          # same as Review-1
EOT_STEPS, EOT_SAMPLES = 60, 4          # 60 steps x 4 random transforms per step
print(list(TRANSFORMS))

## 3 · Part A + B — protect every face with PGD and with EoT-PGD, then "share" it

For each face and ε we build both protected versions, then push each through every
transform and record whether the fake is still broken. Results are cached to CSV,
so if Colab disconnects you can reload them instead of re-running.

In [ ]:
def run(method, eps):
    rows = []
    for i in tqdm(range(N_IMAGES), desc=f"{method} eps={round(eps*255)}/255"):
        x, model = face_and_model(i)
        t0 = time.time()
        if method == "PGD":
            xp = pgd_disrupt(model, x, eps=eps, steps=PGD_STEPS)
        else:
            xp = eot_pgd_disrupt(model, x, eps=eps, steps=EOT_STEPS, n_samples=EOT_SAMPLES, seed=i)
        secs = time.time() - t0
        xp8 = quantize_8bit(xp)
        base = {"method": method, "eps": f"{round(eps*255)}/255", "image": test_ids[i],
                "psnr": psnr(x, xp8), "ssim": ssim(x, xp8), "seconds": secs}
        for r in survival_rows(model, x, xp):
            rows.append({**base, **r})
    return pd.DataFrame(rows)

CACHE = "results/robustness_all.csv"
if os.path.exists(CACHE):
    rob = pd.read_csv(CACHE)
    print("Loaded cached results from", CACHE)
else:
    rob = pd.concat([run(m, e) for e in EPS_LIST for m in ("PGD", "EoT-PGD")], ignore_index=True)
    rob.to_csv(CACHE, index=False)
rob.head()

## 4 · Part C — Review-2 tables

In [ ]:
# (a) invisibility + cost: is EoT still invisible, and how much slower is it?
cost = (rob.drop_duplicates(["method", "eps", "image"])
           .groupby(["eps", "method"], sort=False)[["psnr", "ssim", "seconds"]].mean().round(3))
display(cost)

# (b) protection survival: success rate (%) after each sharing transform
order = list(TRANSFORMS)
succ = (rob.pivot_table(index="transform", columns=["eps", "method"], values="success", aggfunc="mean")
           .reindex(index=order, columns=COLS) * 100).round(0)
display(succ)

# (c) mean output MSE after each transform (higher = fake more broken)
mse = rob.pivot_table(index="transform", columns=["eps", "method"], values="output_mse").reindex(index=order, columns=COLS).round(3)
display(mse)

table = (rob.groupby(["eps", "method", "transform"], sort=False)
            .agg(Output_MSE=("output_mse", "mean"), Success=("success", "mean")).reset_index())
table["Success"] = (table["Success"] * 100).round(0).astype(int).astype(str) + "%"
table["Output_MSE"] = table["Output_MSE"].round(3)
table.to_csv("results/review2_robustness_table.csv", index=False)
try:
    print(succ.to_markdown())
except ImportError:
    print(succ.to_string())

## 5 · Figure — protection survival, PGD vs EoT-PGD

In [ ]:
fig, axes = plt.subplots(1, len(EPS_LIST), figsize=(7 * len(EPS_LIST), 4.6), sharey=True)
axes = np.atleast_1d(axes)
colors = {"PGD": "#9aa5b1", "EoT-PGD": "#1f4e8c"}
for ax, eps in zip(axes, [f"{round(e*255)}/255" for e in EPS_LIST]):
    y = np.arange(len(order)); h = 0.38
    for k, m in enumerate(("PGD", "EoT-PGD")):
        vals = succ[(eps, m)].values
        ax.barh(y + (k - 0.5) * h, vals, height=h, color=colors[m], label=m)
    ax.set_yticks(y); ax.set_yticklabels(order); ax.invert_yaxis()
    ax.set_xlim(0, 105); ax.set_xlabel("Faces still protected after sharing (%)")
    ax.set_title(f"ε = {eps}"); ax.grid(axis="x", alpha=0.25)
axes[0].legend(frameon=False, loc="lower right")
fig.suptitle("Does the protection survive sharing?  (StarGAN, 20 CelebA faces)", fontsize=13)
plt.tight_layout()
plt.savefig("docs/figures/robustness_success.png", dpi=200, bbox_inches="tight")
plt.show()

## 6 · Figure — what the deepfaker actually gets after WhatsApp

Row per face: original → its WhatsApp-shared fake · PGD-protected photo after WhatsApp →
its fake · EoT-protected photo after WhatsApp → its fake.

In [ ]:
FIG_EPS, EDIT, N_SHOW = 8/255, 1, 3
share = TRANSFORMS["WhatsApp-like"]
def img(t): return (t if t.dim() == 3 else t[0]).detach().clamp(0, 1).permute(1, 2, 0).cpu().numpy()

titles = ["Original", "Fake (unprotected, shared)", "PGD-protected, shared", "Fake of it",
          "EoT-protected, shared", "Fake of it"]
fig, ax = plt.subplots(N_SHOW, 6, figsize=(18, 3.2 * N_SHOW))
ax = np.atleast_2d(ax)
for r in range(N_SHOW):
    x, model = face_and_model(r)
    xp = quantize_8bit(pgd_disrupt(model, x, eps=FIG_EPS, steps=PGD_STEPS))
    xe = quantize_8bit(eot_pgd_disrupt(model, x, eps=FIG_EPS, steps=EOT_STEPS, n_samples=EOT_SAMPLES, seed=r))
    with torch.no_grad():
        sc, sp, se = share(quantize_8bit(x)), share(xp), share(xe)
        fc, fp, fe = model(sc)[EDIT], model(sp)[EDIT], model(se)[EDIT]
    for c, p in enumerate([x, fc, sp, fp, se, fe]):
        ax[r, c].imshow(img(p)); ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
        if r == 0: ax[r, c].set_title(titles[c], fontsize=10)
    ax[r, 3].set_xlabel(f"MSE {torch.mean((fp - fc) ** 2).item():.3f}", fontsize=9)
    ax[r, 5].set_xlabel(f"MSE {torch.mean((fe - fc) ** 2).item():.3f}", fontsize=9)
fig.suptitle(f"After WhatsApp-like sharing (ε = {round(FIG_EPS*255)}/255, edit: {EDIT_NAMES[EDIT]})", fontsize=13)
plt.tight_layout()
plt.savefig("docs/figures/whatsapp_before_after.png", dpi=200, bbox_inches="tight")
plt.show()

## 7 · Part D — the REAL WhatsApp / Instagram test (task 2.7)

Simulations are approximations. This part is the proof for the viva.

**Step 1** (cell below) saves 5 faces × {clean, PGD, EoT} as PNGs and zips them.
**Step 2** (on your phone):
1. Download `real_share_upload.zip` from Drive, unzip on the phone.
2. **WhatsApp:** send all 15 images to a teammate *as photos* (NOT as documents — documents
   are not compressed). Keep "Photo upload quality = Standard" (Settings → Storage and data).
   The teammate saves the received images.
3. **Instagram:** post them to a private test account (or send by DM), then download them back.
4. Rename the downloaded files back to the **same names** (`clean_00.jpg`, `pgd_00.jpg`,
   `eot_00.jpg`, …) and upload them to `MyDrive/immunize/real_share/whatsapp/` and
   `MyDrive/immunize/real_share/instagram/`.

**Step 3** — run the evaluation cell after that.

In [ ]:
import shutil
from torchvision.utils import save_image
N_REAL, REAL_EPS = 5, 8/255
os.makedirs("results/real_share_upload", exist_ok=True)
for i in tqdm(range(N_REAL), desc="export"):
    x, model = face_and_model(i)
    save_image(x, f"results/real_share_upload/clean_{i:02d}.png")
    save_image(pgd_disrupt(model, x, eps=REAL_EPS, steps=PGD_STEPS), f"results/real_share_upload/pgd_{i:02d}.png")
    save_image(eot_pgd_disrupt(model, x, eps=REAL_EPS, steps=EOT_STEPS, n_samples=EOT_SAMPLES, seed=i),
               f"results/real_share_upload/eot_{i:02d}.png")
shutil.make_archive("results/real_share_upload", "zip", "results/real_share_upload")
print("Created results/real_share_upload.zip — copy to Drive in the last cell")

In [ ]:
# Step 3 — evaluate the photos that really went through the apps.
from google.colab import drive
drive.mount("/content/drive")
REAL_DIR = "/content/drive/MyDrive/immunize/real_share"

def load_shared(path):
    im = Image.open(path).convert("RGB")
    t = T.ToTensor()(im).unsqueeze(0)
    # apps may resize (Instagram upscales small photos) — the attacker resizes back to 256
    t = torch.nn.functional.interpolate(t, size=(256, 256), mode="bilinear", antialias=True, align_corners=False)
    return t.to(device)

def find(folder, stem):
    hits = glob.glob(f"{folder}/{stem}.*")
    return hits[0] if hits else None

real_rows = []
for app in ("whatsapp", "instagram"):
    folder = f"{REAL_DIR}/{app}"
    if not os.path.isdir(folder):
        print("skip (folder not uploaded yet):", folder); continue
    for i in range(N_REAL):
        _, model = face_and_model(i)
        pc = find(folder, f"clean_{i:02d}")
        if pc is None: continue
        with torch.no_grad():
            ref = model(load_shared(pc))
            for method, stem in (("PGD", "pgd"), ("EoT-PGD", "eot")):
                p = find(folder, f"{stem}_{i:02d}")
                if p is None: continue
                mse = torch.mean((model(load_shared(p)) - ref) ** 2).item()
                real_rows.append({"app": app, "method": method, "image": i,
                                  "output_mse": mse, "success": mse >= MSE_SUCCESS})
if real_rows:
    real = pd.DataFrame(real_rows)
    real.to_csv("results/real_share_results.csv", index=False)
    display(real.groupby(["app", "method"]).agg(output_mse=("output_mse", "mean"),
                                                success_rate=("success", "mean")).round(3))

## 8 · Save everything to Google Drive (share this folder with the team)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
shutil.copytree("results", "/content/drive/MyDrive/immunize/results", dirs_exist_ok=True)
shutil.copytree("docs/figures", "/content/drive/MyDrive/immunize/figures", dirs_exist_ok=True)
print("Saved results/ and figures to MyDrive/immunize/")